# Prompt cache through the Anthropic api

Prompt cache allows you para store e reuse context within your prompt. este makes isso mais practical para include additional information in your prompt—such as detailed instructions e example responses—qual help melhorar every resposta Claude generates.

In addition, by fully leveraging prompt cache within your prompt, you can reduce latency by >2x e costs up para 90%. este can generate significant savings quando building solutions aquele involve repetitive tasks ao redor detailed book_content.

In este cookbook, we will demonstrate como para use prompt cache in a single turn e across a multi-turn conversation. 


## Configuração

primeiro, let's definir up our environment com the necessary imports e initializations:

In [3]:
%pip install anthropic bs4 --quiet

NOTA: you may need to restart the kernel to use updated packages.


In [4]:
importar anthropic
importar time
importar requests
from bs4 importar BeautifulSoup

client = anthropic.Anthropic()
MODEL_NAME = "claude-3-5-sonnet-20241022"

agora let's fetch alguns texto content para use in our Exemplos. We'll use the texto de Pride e Prejudice by Jane Austen qual is ao redor ~187,000 tokens long.

In [5]:
def fetch_article_content(url):
    response = requests.obter(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Remove script and style elements
    para script in soup(["script", "style"]):
        script.decompose()
    
    # obter text
    text = soup.get_text()
    
    # parar into lines and remove leading and trailing space on each
    lines = (line.limpar() para line in text.splitlines())
    # parar multi-headlines into a line each
    chunks = (phrase.limpar() para line in lines para phrase in line.dividir("  "))
    # Drop blank lines
    text = '\n'.juntar(chunk para chunk in chunks se chunk)
    
    retornar text

# Fetch the content of the article
book_url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
book_content = fetch_article_content(book_url)

imprimir(f"Fetched {len(book_content)} characters from the book.")
imprimir("First 500 characters:")
imprimir(book_content[:500])

Fetched 737525 characters from the book.
First 500 characters:
The Project Gutenberg eBook of Pride and Prejudice
This ebook is para the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy isso, give isso away or re-use isso under the terms
of the Project Gutenberg licença included with this ebook or online
at www.gutenberg.org. se you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.
Title:


## Example 1: Single turn

Let's demonstrate prompt cache com a large document, comparing the desempenho e cost entre cached e non-cached api calls.

### Part 1: Non-cached api Call

primeiro, let's make a non-cached api call. este will carregar the prompt into the cache so aquele our subsequent cached api calls can benefit de the prompt cache.

We will ask para a short saída texto para keep the saída resposta tempo low since the benefit of prompt cache applies only para the entrada processing tempo.

In [6]:
def make_non_cached_api_call():
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "tipo": "text",
                    "text": "<book>" + book_content + "</book>",
                    "cache_control": {"tipo": "ephemeral"}
                },
                {
                    "tipo": "text",
                    "text": "What is the title of this book? Only output the title."
                }
            ]
        }
    ]

    start_time = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        messages=messages,
        extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"}

    )
    end_time = time.time()

    retornar response, end_time - start_time

non_cached_response, non_cached_time = make_non_cached_api_call()

imprimir(f"Non-cached api call time: {non_cached_time:.2f} seconds")
imprimir(f"Non-cached api call entrada tokens: {non_cached_response.Uso.input_tokens}")
imprimir(f"Non-cached api call output tokens: {non_cached_response.Uso.output_tokens}")

imprimir("\nSummary (non-cached):")
imprimir(non_cached_response.content)

Non-cached api call time: 20.37 seconds
Non-cached api call entrada tokens: 17
Non-cached api call output tokens: 8

Summary (non-cached):
[TextBlock(text='Pride and Prejudice', tipo='text')]


### Part 2: Cached api Call

agora, let's make a cached api call. I'll adicionar in the "cache_control": {"tipo": "ephemeral"} attribute para the content objeto e adicionar the "prompt-cache-2024-07-31" beta header para the requisição. este will enable prompt cache para este api call.

para keep the saída latency constant, we will ask Claude the same question as antes. Nota aquele este question is não part of the cached content.

In [7]:
def make_cached_api_call():
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "tipo": "text",
                    "text": "<book>" + book_content + "</book>",
                    "cache_control": {"tipo": "ephemeral"}
                },
                {
                    "tipo": "text",
                    "text": "What is the title of this book? Only output the title."
                }
            ]
        }
    ]

    start_time = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        messages=messages,
        extra_headers={"anthropic-beta": "prompt-caching-2024-07-31"}
    )
    end_time = time.time()

    retornar response, end_time - start_time

cached_response, cached_time = make_cached_api_call()

imprimir(f"Cached api call time: {cached_time:.2f} seconds")
imprimir(f"Cached api call entrada tokens: {cached_response.Uso.input_tokens}")
imprimir(f"Cached api call output tokens: {cached_response.Uso.output_tokens}")

imprimir("\nSummary (cached):")
imprimir(cached_response.content)

Cached api call time: 2.92 seconds
Cached api call entrada tokens: 17
Cached api call output tokens: 8

Summary (cached):
[TextBlock(text='Pride and Prejudice', tipo='text')]


As you can see, the cached api call only took 3.64 seconds total compared para 21.44 seconds para the non-cached api call. este is a significant improvement in overall latency due para cache.

## Example 2: Multi-turn Conversation com Incremental cache

agora, let's look at a multi-turn conversation onde we adicionar cache breakpoints as the conversation progresses.

In [8]:
classe ConversationHistory:
    def __init__(self):
        # Initialize an empty list to store conversation turns
        self.turns = []

    def add_turn_assistant(self, content):
        # Add an assistant's turn to the conversation history
        self.turns.anexar({
            "role": "assistant",
            "content": [
                {
                    "tipo": "text",
                    "text": content
                }
            ]
        })

    def add_turn_user(self, content):
        # Add a user's turn to the conversation history
        self.turns.anexar({
            "role": "user",
            "content": [
                {
                    "tipo": "text",
                    "text": content
                }
            ]
        })

    def get_turns(self):
        # Retrieve conversation turns with specific formatting
        result = []
        user_turns_processed = 0
        # Iterate through turns in reverse order
        para turn in reversed(self.turns):
            se turn["role"] == "user" and user_turns_processed < 1:
                # Add the last user turn with ephemeral cache control
                result.anexar({
                    "role": "user",
                    "content": [
                        {
                            "tipo": "text",
                            "text": turn["content"][0]["text"],
                            "cache_control": {"tipo": "ephemeral"}
                        }
                    ]
                })
                user_turns_processed += 1
            senão:
                # Add other turns as they are
                result.anexar(turn)
        # retornar the turns in the original order
        retornar list(reversed(result))

# Initialize the conversation history
conversation_history = ConversationHistory()

# System message containing the book content
# NOTA: 'book_content' should be defined elsewhere in the code
system_message = f"<file_contents> {book_content} </file_contents>"

# Predefined questions para our simulation
questions = [
    "What is the title of this novel?",
    "Who are Mr. and Mrs. Bennet?",
    "What is Netherfield Park?",
    "What is the principal theme of this novel?"
]

def simulate_conversation():
    para i, question in enumerate(questions, 1):
        imprimir(f"\nTurn {i}:")
        imprimir(f"User: {question}")
        
        # Add user entrada to conversation history
        conversation_history.add_turn_user(question)

        # Record the start time para performance measurement
        start_time = time.time()

        # Make an api call to the assistant
        response = client.messages.create(
            model=MODEL_NAME,
            extra_headers={
              "anthropic-beta": "prompt-caching-2024-07-31"
            },
            max_tokens=300,
            system=[
                {"tipo": "text", "text": system_message, "cache_control": {"tipo": "ephemeral"}},
            ],
            messages=conversation_history.get_turns(),
        )

        # Record the end time
        end_time = time.time()

        # Extract the assistant's reply
        assistant_reply = response.content[0].text
        imprimir(f"Assistant: {assistant_reply}")

        # imprimir token Uso information
        input_tokens = response.Uso.input_tokens
        output_tokens = response.Uso.output_tokens
        input_tokens_cache_read = getattr(response.Uso, 'cache_read_input_tokens', '---')
        input_tokens_cache_create = getattr(response.Uso, 'cache_creation_input_tokens', '---')
        imprimir(f"User entrada tokens: {input_tokens}")
        imprimir(f"Output tokens: {output_tokens}")
        imprimir(f"entrada tokens (cache ler): {input_tokens_cache_read}")
        imprimir(f"entrada tokens (cache escrever): {input_tokens_cache_create}")

        # Calculate and imprimir the elapsed time
        elapsed_time = end_time - start_time

        # Calculate the percentage of entrada prompt cached
        total_input_tokens = input_tokens + (int(input_tokens_cache_read) se input_tokens_cache_read != '---' senão 0)
        percentage_cached = (int(input_tokens_cache_read) / total_input_tokens * 100 se input_tokens_cache_read != '---' and total_input_tokens > 0 senão 0)

        imprimir(f"{percentage_cached:.1f}% of entrada prompt cached ({total_input_tokens} tokens)")
        imprimir(f"Time taken: {elapsed_time:.2f} seconds")

        # Add assistant's reply to conversation history
        conversation_history.add_turn_assistant(assistant_reply)

# Run the simulated conversation
simulate_conversation()


Turn 1:
User: What is the title of this novel?
Assistant: The title of this novel is "Pride and Prejudice" by Jane Austen.
User entrada tokens: 4
Output tokens: 22
entrada tokens (cache ler): 0
entrada tokens (cache escrever): 187354
0.0% of entrada prompt cached (4 tokens)
Time taken: 20.37 seconds

Turn 2:
User: Who are Mr. and Mrs. Bennet?
Assistant: Mr. and Mrs. Bennet are the parents of five daughters (Jane, Elizabeth, Mary, Kitty, and Lydia) in Pride and Prejudice. 

Mr. Bennet is an intelligent but detached father who often retreats to his library to avoid his wife's dramatics. He has a satirical wit and tends to be amused by the follies of others, including his own family members. He shows particular fondness para his second daughter Elizabeth, who shares his sharp mind and wit.

Mrs. Bennet is a woman primarily focused on getting her five daughters married to wealthy men. She is described as having "poor nerves" and is often anxious, dramatic, and somewhat foolish. Her princi

As you can see in Este exemplo, resposta times decreased de quase 24 seconds para just 7-11 seconds depois the initial cache Configuração, enquanto maintaining the same level of quality across the answers. maioria of este remaining latency is due para the tempo isso takes para generate the resposta, qual is não affected by prompt cache.

e since quase 100% of entrada tokens were cached in subsequent turns as we kept adjusting the cache breakpoints, we were able para ler the próximo usuário message quase instantly.